In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [5]:
file_path = "/Volumes/Engineering/Database/HMCorp/java/train.jsonl"
hmcorp = pd.read_json(file_path, lines=True)

In [6]:
df = pd.read_csv("../data/hmcorp_xml.csv")

print("Shape of output:", df.shape)

print(df.columns)


Shape of output: (355736, 5)
Index(['id', 'code', 'label', 'language', 'xml'], dtype='str')


In [7]:
df.head()

,id,code,label,language,xml
0,gj235730,public Object readValue(Object value) {\n /...,1,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."
1,gj235730,private OptionKindAndValue readKindAndValue() ...,0,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."
2,gj164536,public <U> CompletableFuture<U> thenComposeAsy...,1,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."
3,gj164536,public <TContinuationResult> Task<TContinuatio...,0,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."
4,gj096930,public String getUserAgent() {\n String jav...,1,java,"<?xml version=""1.0"" encoding=""UTF-8"" standalon..."


In [8]:
unique_ids = df["id"].unique()
print(unique_ids)

<StringArray>
['gj235730', 'gj164536', 'gj096930', 'gj008079', 'gj166166', 'gj178131',
 'gj097050', 'gj095207', 'gj160320', 'gj017145',
 ...
 'gj058951', 'gj181773', 'gj196606', 'gj107861', 'gj136229', 'gj106691',
 'gj163801', 'gj160435', 'gj096701', 'gj269528']
Length: 177868, dtype: str


In [9]:
import re

_FILENAME_ATTR = re.compile(r'\s+filename="[^"]*"')


def strip_xml_leak(xml: str) -> str:
    return _FILENAME_ATTR.sub("", xml)


In [16]:
# Allow the column width to expand infinitely
pd.set_option('display.max_colwidth', None)

# Print the full string
print(df["xml"][2])

<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<unit xmlns="http://www.srcML.org/srcML/src" revision="1.0.0" language="Java" filename="temp/gj164536_human.java"><function><type><specifier>public</specifier> <parameter_list type="generic">&lt;<parameter><name>U</name></parameter>&gt;</parameter_list> <name><name>CompletableFuture</name><argument_list type="generic">&lt;<argument><name>U</name></argument>&gt;</argument_list></name></type> <name>thenComposeAsync</name><parameter_list>(<parameter><decl><type><name><name>Function</name><argument_list type="generic">&lt;<argument><name>?</name> <super>super <name>T</name></super></argument>, <argument><name>?</name> <extends>extends <name><name>CompletionStage</name><argument_list type="generic">&lt;<argument><name>U</name></argument>&gt;</argument_list></name></extends></argument>&gt;</argument_list></name></type> <name>fn</name></decl></parameter>)</parameter_list> <block>{<block_content>
    <return>return <expr><name><name>Compl

In [17]:
df["xml"] = df["xml"].apply(strip_xml_leak)

In [18]:
# Print the full string
print(df["xml"][2])

<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<unit xmlns="http://www.srcML.org/srcML/src" revision="1.0.0" language="Java"><function><type><specifier>public</specifier> <parameter_list type="generic">&lt;<parameter><name>U</name></parameter>&gt;</parameter_list> <name><name>CompletableFuture</name><argument_list type="generic">&lt;<argument><name>U</name></argument>&gt;</argument_list></name></type> <name>thenComposeAsync</name><parameter_list>(<parameter><decl><type><name><name>Function</name><argument_list type="generic">&lt;<argument><name>?</name> <super>super <name>T</name></super></argument>, <argument><name>?</name> <extends>extends <name><name>CompletionStage</name><argument_list type="generic">&lt;<argument><name>U</name></argument>&gt;</argument_list></name></extends></argument>&gt;</argument_list></name></type> <name>fn</name></decl></parameter>)</parameter_list> <block>{<block_content>
    <return>return <expr><name><name>CompletableFuture</name><operator>.</oper

In [26]:
def findKeywords(xml):
    xml = xml.lower()
    if "human" in xml:
        return "human"
    # if "ai" in xml:
    #     return "ai"
    return None

df["keyword"] = df["xml"].apply(findKeywords)

count = df["keyword"].notna().sum()
print(len(df))
print(count)


355736
135


In [31]:
flagged_df = df[df["keyword"].notna()]

In [32]:
df = df[df["keyword"].isna()].copy()
df = df.drop(index=flagged_df.index).copy()

In [33]:
print(df["keyword"].notna().sum())
print(df.shape)

0
(355601, 6)


In [34]:
df.drop(columns=["keyword"], inplace=True)
df.to_csv("../data/hmcorp_xml_2.csv", index=False)

In [35]:
df2 = pd.read_csv("../data/hmcorp_xml_2.csv")

In [36]:
df2.shape

(355601, 5)

In [39]:
def findKeywords(xml):
    xml = xml.lower()
    if "human" in xml:
        return "human"
    # if "ai" in xml:
    #     return "ai"
    return None

df2["keyword"] = df2["xml"].apply(findKeywords)
count = df2["keyword"].notna().sum()
print(count)

0
